# S01 — Validate the infraBSU and Centaur response

This is a **supporting methods-validation notebook**, not a production step.

It preserves the detailed investigations formerly embedded in Notebook 01:

- nominal infraBSU and Centaur NRL response construction;
- 40 Vpp versus 1 Vpp digitizer configurations;
- empirical Antelope calibration versus nominal sensitivity;
- stage-by-stage response inspection;
- frequency-response comparison;
- sensitivity-only pressure conversion;
- optional full-response deconvolution tests.

Its purpose is to document and test the response model. It must not overwrite the
authoritative waveform products created by
`01_correct_bchh_instrument_response_refactored.ipynb`.


## 1. Imports and validation configuration


In [ ]:
from __future__ import annotations

import copy
import json
import os
from pathlib import Path
from typing import Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from obspy import Stream, UTCDateTime, read, read_inventory
from obspy.clients.nrl import NRL
from obspy.core.inventory import Channel, Inventory, Network, Site, Station
from obspy.core.inventory.response import InstrumentSensitivity, Response

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})


In [ ]:
# Project paths
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

MINISEED_DIR = Path(
    "/Volumes/haldata/KSC/beforePASSCAL/EVENTS/20160901_SpaceXplosion"
)
STATIONXML_FILE = PROJECT_ROOT / "metadata" / "KSC.xml"
CORRECTED_STATIONXML = OUTPUT_DIR / "bchh_event_inventory_validation_copy.xml"

# Antelope CSS3.0 tables from the original 2016 processing workflow.
# Edit these paths if the files are stored elsewhere.
CSS_CALIBRATION_FILE = PROJECT_ROOT / "metadata" / "sitedb.calibration"
CSS_SITE_FILE = PROJECT_ROOT / "metadata" / "sitedb.site"
CSS_SITECHAN_FILE = PROJECT_ROOT / "metadata" / "sitedb.sitechan"

# Previously downloaded sensor-only infraBSU StationXML template.
INFRABSU_TEMPLATE_FILE = (
    PROJECT_ROOT.parent.parent
    / "flovopy"
    / "flovopy"
    / "stationmetadata"
    / "stationxml_templates"
    / "infraBSU_sensor.xml"
)

# None means use ObsPy's remote NRL for the Centaur stages.
NRL_PATH = None

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "response_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_CACHE = OUTPUT_DIR / "bchh_raw_event_window.mseed"
CORRECTED_MSEED = OUTPUT_DIR / "bchh_corrected_antelope_calibration.mseed"
ORIGINAL_EVENT_STATIONXML = OUTPUT_DIR / "bchh_event_inventory_original.xml"
CALIBRATED_EVENT_STATIONXML = OUTPUT_DIR / "bchh_event_inventory_antelope_calibrated.xml"
CHANNEL_METADATA_CSV = OUTPUT_DIR / "bchh_channel_metadata.csv"
ACTIVE_CALIBRATION_CSV = OUTPUT_DIR / "bchh_active_antelope_calibration.csv"
CORRECTION_SUMMARY_CSV = OUTPUT_DIR / "bchh_correction_summary.csv"
NRL_COMPARISON_CSV = OUTPUT_DIR / "infrabsu_nrl_response_comparison.csv"
PROCESSING_JSON = OUTPUT_DIR / "bchh_response_processing.json"

OVERWRITE_RAW_CACHE = False
OVERWRITE_CORRECTED = True

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")
READ_START = EXPLOSION_TIME - 180.0
READ_END = EXPLOSION_TIME + 1800.0

# Quiet interval used to remove each pressure channel's raw-count DC offset.
PRESSURE_BASELINE_START = EXPLOSION_TIME - 20.0
PRESSURE_BASELINE_END = EXPLOSION_TIME - 2.0

SEISMIC_PRE_FILT = (0.02, 0.05, 80.0, 100.0)
SEISMIC_WATER_LEVEL_DB = 60.0

BCHH_NETWORK = "1R"
BCHH_STATION = "BCHH"
BCHH_LOCATION = "10"

WAVEFORM_TO_XML_CHANNEL = {
    "HD1": "DD1",
    "HD2": "DD2",
    "HD3": "DD3",
    "HHE": "DHE",
    "HHN": "DHN",
    "HHZ": "DHZ",
}
XML_TO_WAVEFORM_CHANNEL = {v: k for k, v in WAVEFORM_TO_XML_CHANNEL.items()}
CHANNEL_ORDER = list(WAVEFORM_TO_XML_CHANNEL)

for required_path, label in [
    (STATIONXML_FILE, "event StationXML"),
    (CSS_CALIBRATION_FILE, "Antelope calibration table"),
    (INFRABSU_TEMPLATE_FILE, "infraBSU sensor template"),
]:
    if not required_path.exists():
        print(f"WARNING: {label} not found at {required_path}")


## 2. Load and subset the event inventory

Coordinates, elevations, channel orientations, sample rates, and responses are all taken from the active StationXML epoch. No station coordinates are hardcoded.


In [ ]:
def load_event_inventory(
    stationxml_file: Path,
    *,
    network: str,
    station: str,
    location: str,
    starttime: UTCDateTime,
    endtime: UTCDateTime,
) -> Inventory:
    """Load the StationXML and retain only active BCHH channel epochs."""
    if not stationxml_file.exists():
        raise FileNotFoundError(stationxml_file)

    inventory = read_inventory(str(stationxml_file))
    selected = inventory.select(
        network=network,
        station=station,
        location=location,
        time=starttime,
    )

    # Retain only channels that overlap the requested time interval.
    keep = set(WAVEFORM_TO_XML_CHANNEL.values())
    selected = copy.deepcopy(selected)
    for net in selected:
        for sta in net:
            sta.channels = [
                cha for cha in sta.channels
                if cha.code in keep
                and (cha.start_date is None or cha.start_date <= endtime)
                and (cha.end_date is None or cha.end_date >= starttime)
            ]

    if not any(sta.channels for net in selected for sta in net):
        raise ValueError("No active BCHH response channels found in StationXML")

    return selected


def inventory_channels_to_dataframe(inventory: Inventory) -> pd.DataFrame:
    """Flatten channel coordinates and response sensitivities into a table."""
    rows = []
    for net in inventory:
        for sta in net:
            for cha in sta:
                response = cha.response
                sensitivity = getattr(response, "instrument_sensitivity", None)
                rows.append({
                    "seed_id": f"{net.code}.{sta.code}.{cha.location_code}.{cha.code}",
                    "network": net.code,
                    "station": sta.code,
                    "location": cha.location_code,
                    "channel": cha.code,
                    "waveform_channel": XML_TO_WAVEFORM_CHANNEL.get(cha.code),
                    "latitude": float(cha.latitude),
                    "longitude": float(cha.longitude),
                    "elevation_m": float(cha.elevation),
                    "depth_m": float(cha.depth),
                    "azimuth_deg": None if cha.azimuth is None else float(cha.azimuth),
                    "dip_deg": None if cha.dip is None else float(cha.dip),
                    "sample_rate_hz": None if cha.sample_rate is None else float(cha.sample_rate),
                    "sensitivity": None if sensitivity is None else float(sensitivity.value),
                    "sensitivity_frequency_hz": None if sensitivity is None else float(sensitivity.frequency),
                    "input_units": None if sensitivity is None else str(sensitivity.input_units),
                    "output_units": None if sensitivity is None else str(sensitivity.output_units),
                })
    return pd.DataFrame(rows).sort_values("channel").reset_index(drop=True)


inventory_event = load_event_inventory(
    STATIONXML_FILE,
    network=BCHH_NETWORK,
    station=BCHH_STATION,
    location=BCHH_LOCATION,
    starttime=READ_START,
    endtime=READ_END,
)

channel_metadata_df = inventory_channels_to_dataframe(inventory_event)
display(channel_metadata_df)

inventory_event.write(str(CORRECTED_STATIONXML), format="STATIONXML", validate=True)
channel_metadata_df.to_csv(CHANNEL_METADATA_CSV, index=False)
print(f"Saved event inventory: {CORRECTED_STATIONXML}")
print(f"Saved channel coordinates: {CHANNEL_METADATA_CSV}")


## 3. Read and cache the raw BCHH stream


In [ ]:
def read_bchh_stream(
    miniseed_dir: Path,
    cache_file: Path,
    starttime: UTCDateTime,
    endtime: UTCDateTime,
    *,
    overwrite_cache: bool = False,
) -> Stream:
    """Read BCHH day files, trim, merge, and optionally cache the raw counts."""
    if cache_file.exists() and not overwrite_cache:
        st = read(str(cache_file))
        st.trim(starttime, endtime, pad=False)
        return st

    patterns = ["*.20160901T000000", "*.mseed", "*.msd", "*.miniseed"]
    files = []
    for pattern in patterns:
        files.extend(miniseed_dir.glob(pattern))
    files = sorted(set(files))

    if not files:
        raise FileNotFoundError(f"No miniSEED files found in {miniseed_dir}")

    st = Stream()
    for filename in files:
        st += read(str(filename))

    st.trim(starttime, endtime, pad=False)
    st.merge(method=1, fill_value="interpolate")
    st.sort(keys=["station", "channel"])
    st.write(str(cache_file), format="MSEED")
    return st


st_raw = read_bchh_stream(
    MINISEED_DIR,
    RAW_CACHE,
    READ_START,
    READ_END,
    overwrite_cache=OVERWRITE_RAW_CACHE,
)

print(st_raw)
for tr in st_raw:
    print(tr.id, tr.stats.starttime, tr.stats.endtime, tr.stats.sampling_rate, tr.stats.npts)


## 4. Read the original Antelope calibration table

For pressure channels, Antelope `calib` is interpreted as the multiplier from raw counts to physical units. Thus the event-epoch values are directly in **Pa/count**. These channel-specific empirical values are authoritative for the Fireball processing.


In [ ]:
def read_css_calibration_table(path: Path | str) -> pd.DataFrame:
    """Read the fields needed from an Antelope CSS3.0 calibration table.

    The instrument and recorder descriptions contain spaces, so fields are
    extracted from both ends of each whitespace-delimited row:
      sta, chan, time, endtime, ..., calib, calper, tshift, units, lddate
    """
    rows = []
    path = Path(path)
    for line_number, line in enumerate(path.read_text().splitlines(), start=1):
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        fields = line.split()
        if len(fields) < 10:
            raise ValueError(f"Malformed calibration row {line_number}: {line}")
        rows.append({
            "station": fields[0],
            "css_channel": fields[1],
            "start_epoch": float(fields[2]),
            "end_epoch": float(fields[3]),
            "calib": float(fields[-5]),
            "calper": float(fields[-4]),
            "tshift": float(fields[-3]),
            "units": fields[-2],
            "lddate_epoch": float(fields[-1]),
            "line_number": line_number,
            "raw_line": line,
        })
    return pd.DataFrame(rows)


def active_css_calibrations(
    table: pd.DataFrame,
    *,
    station: str,
    event_time: UTCDateTime,
    waveform_channels: Iterable[str],
) -> pd.DataFrame:
    """Select exactly one active calibration row for each waveform channel."""
    event_epoch = float(event_time.timestamp)
    selected_rows = []

    for waveform_channel in waveform_channels:
        css_channel = f"{waveform_channel}_00"
        matches = table[
            (table["station"] == station)
            & (table["css_channel"] == css_channel)
            & (table["start_epoch"] <= event_epoch)
            & (table["end_epoch"] >= event_epoch)
        ]
        if len(matches) != 1:
            raise ValueError(
                f"Expected one active calibration for {station}.{css_channel} "
                f"at {event_time}; found {len(matches)}"
            )
        selected_rows.append(matches.iloc[0])

    result = pd.DataFrame(selected_rows).reset_index(drop=True)
    result["waveform_channel"] = result["css_channel"].str.replace(
        "_00", "", regex=False
    )
    result["count_per_unit"] = 1.0 / result["calib"]
    return result


css_calibration_df = read_css_calibration_table(CSS_CALIBRATION_FILE)
active_pressure_calibration_df = active_css_calibrations(
    css_calibration_df,
    station=BCHH_STATION,
    event_time=EXPLOSION_TIME,
    waveform_channels=["HD1", "HD2", "HD3"],
)

print("Active empirical pressure calibrations:")
display(
    active_pressure_calibration_df[
        [
            "waveform_channel",
            "start_epoch",
            "end_epoch",
            "calib",
            "units",
            "count_per_unit",
        ]
    ]
)

pressure_calib_pa_per_count = dict(
    zip(
        active_pressure_calibration_df["waveform_channel"],
        active_pressure_calibration_df["calib"],
    )
)

for channel, value in pressure_calib_pa_per_count.items():
    print(f"{channel}: {value:g} Pa/count ({1.0/value:g} count/Pa)")


## 5. Construct an in-memory empirical response for validation

The following helper code creates corrected traces and an empirically scaled
inventory for comparison only. The authoritative files are written by Notebook 01.


## 4. Production correction using the event StationXML

The pressure and seismic channels are treated differently:

- **infraBSU (`HD*`)**: counts are divided by the StationXML overall sensitivity. The response is approximately flat over the analysis band, and this avoids unstable inversion near zero frequency. A pre-event median is then removed to eliminate the digitizer DC offset.
- **Trillium Compact (`HH*`)**: the complete response is removed to velocity using a pre-filter and finite water level.

The original waveform IDs are restored after response lookup.


In [ ]:
def remap_trace_to_inventory_id(
    trace,
    *,
    inventory_network: str,
    inventory_station: str,
    inventory_location: str,
):
    """Copy a waveform trace and temporarily assign its StationXML NSLC code."""
    out = trace.copy()
    waveform_channel = out.stats.channel.upper()
    if waveform_channel not in WAVEFORM_TO_XML_CHANNEL:
        raise ValueError(f"No StationXML mapping defined for {trace.id}")

    original_codes = {
        "network": out.stats.network,
        "station": out.stats.station,
        "location": out.stats.location,
        "channel": out.stats.channel,
    }
    out.stats.network = inventory_network
    out.stats.station = inventory_station
    out.stats.location = inventory_location
    out.stats.channel = WAVEFORM_TO_XML_CHANNEL[waveform_channel]
    return out, original_codes


def restore_trace_id(trace, original_codes: dict) -> None:
    for key, value in original_codes.items():
        setattr(trace.stats, key, value)


def median_in_window(trace, starttime: UTCDateTime, endtime: UTCDateTime) -> float:
    window = trace.copy().trim(starttime, endtime, pad=False)
    if window.stats.npts == 0:
        raise ValueError(f"No samples in baseline window for {trace.id}")
    return float(np.nanmedian(np.asarray(window.data, dtype=np.float64)))


def correct_bchh_with_antelope_calibration(
    stream: Stream,
    inventory: Inventory,
    *,
    pressure_calib_pa_per_count: dict[str, float],
    inventory_network: str,
    inventory_station: str,
    inventory_location: str,
    pressure_baseline_start: UTCDateTime,
    pressure_baseline_end: UTCDateTime,
    seismic_pre_filt: tuple[float, float, float, float],
    seismic_water_level_db: float,
) -> tuple[Stream, pd.DataFrame]:
    """Correct BCHH pressure with Antelope calib and seismic with StationXML."""
    corrected = Stream()
    rows = []

    for raw_trace in stream:
        xml_trace, original_codes = remap_trace_to_inventory_id(
            raw_trace,
            inventory_network=inventory_network,
            inventory_station=inventory_station,
            inventory_location=inventory_location,
        )
        response = inventory.get_response(xml_trace.id, xml_trace.stats.starttime)
        sensitivity = response.instrument_sensitivity
        waveform_channel = original_codes["channel"].upper()

        if waveform_channel.startswith("HD"):
            if waveform_channel not in pressure_calib_pa_per_count:
                raise KeyError(f"No empirical calibration for {waveform_channel}")

            calib_pa_per_count = float(
                pressure_calib_pa_per_count[waveform_channel]
            )
            out = raw_trace.copy()
            out.data = np.asarray(out.data, dtype=np.float64)

            baseline_counts = median_in_window(
                out,
                pressure_baseline_start,
                pressure_baseline_end,
            )
            out.data = (out.data - baseline_counts) * calib_pa_per_count

            baseline_physical = baseline_counts * calib_pa_per_count
            units = "Pa"
            method = "antelope_calib_after_pre_event_count_baseline"
            effective_count_per_unit = 1.0 / calib_pa_per_count
            pre_filt = None
            water_level = None

        elif waveform_channel.startswith("HH"):
            out = xml_trace.copy()
            out.detrend("demean")
            out.detrend("linear")
            out.taper(max_percentage=0.05, type="cosine")
            out.remove_response(
                inventory=inventory,
                output="VEL",
                pre_filt=seismic_pre_filt,
                water_level=seismic_water_level_db,
                zero_mean=False,
                taper=False,
            )
            restore_trace_id(out, original_codes)

            calib_pa_per_count = np.nan
            baseline_counts = np.nan
            baseline_physical = np.nan
            units = "m/s"
            method = "event_stationxml_full_response_to_velocity"
            effective_count_per_unit = float(sensitivity.value)
            pre_filt = seismic_pre_filt
            water_level = seismic_water_level_db
        else:
            raise ValueError(f"Unsupported waveform channel {raw_trace.id}")

        out.stats.units = units
        corrected += out
        rows.append({
            "waveform_id": raw_trace.id,
            "inventory_id": xml_trace.id,
            "method": method,
            "units": units,
            "antelope_calib_pa_per_count": calib_pa_per_count,
            "effective_count_per_unit": effective_count_per_unit,
            "stationxml_nominal_sensitivity": float(sensitivity.value),
            "stationxml_input_units": str(sensitivity.input_units),
            "stationxml_output_units": str(sensitivity.output_units),
            "removed_baseline_counts": baseline_counts,
            "removed_baseline_physical_units": baseline_physical,
            "pre_filt": None if pre_filt is None else repr(tuple(pre_filt)),
            "water_level_db": water_level,
        })

    corrected.sort(keys=["station", "channel"])
    return corrected, pd.DataFrame(rows)


st_corrected, correction_summary_df = correct_bchh_with_antelope_calibration(
    st_raw,
    inventory_event,
    pressure_calib_pa_per_count=pressure_calib_pa_per_count,
    inventory_network=BCHH_NETWORK,
    inventory_station=BCHH_STATION,
    inventory_location=BCHH_LOCATION,
    pressure_baseline_start=PRESSURE_BASELINE_START,
    pressure_baseline_end=PRESSURE_BASELINE_END,
    seismic_pre_filt=SEISMIC_PRE_FILT,
    seismic_water_level_db=SEISMIC_WATER_LEVEL_DB,
)

display(correction_summary_df)


## 6. Build an Antelope-calibrated response inventory

For each DD channel, the first response-stage gain is rescaled so that the total response sensitivity equals `1 / calib` count/Pa. This preserves the nominal infraBSU pole-zero shape while recording the empirical channel-specific amplitude calibration.


In [ ]:
def make_antelope_calibrated_inventory(
    inventory: Inventory,
    *,
    network: str,
    station: str,
    location: str,
    channel_calib_pa_per_count: dict[str, float],
    waveform_to_xml_channel: dict[str, str],
    time: UTCDateTime,
) -> Inventory:
    """Return a copy of the inventory with empirical pressure calibrations.

    The Antelope calibration is in Pa/count, so the corresponding StationXML
    sensitivity is its reciprocal, in count/Pa.

    The first response-stage gain is rescaled so the complete response has the
    desired empirical overall sensitivity while retaining its pole-zero shape.
    """
    calibrated = copy.deepcopy(inventory)

    for waveform_channel, calib_pa_per_count in (
        channel_calib_pa_per_count.items()
    ):
        if waveform_channel not in waveform_to_xml_channel:
            raise KeyError(
                f"No StationXML channel mapping for {waveform_channel}"
            )

        xml_channel = waveform_to_xml_channel[waveform_channel]

        selected = calibrated.select(
            network=network,
            station=station,
            location=location,
            channel=xml_channel,
            time=time,
        )

        channels = [
            channel
            for selected_network in selected
            for selected_station in selected_network
            for channel in selected_station
        ]

        if len(channels) != 1:
            raise ValueError(
                "Expected exactly one channel for "
                f"{network}.{station}.{location}.{xml_channel}; "
                f"found {len(channels)}"
            )

        channel = channels[0]
        response = channel.response

        if response is None:
            raise ValueError(
                f"No response attached to {channel.code}"
            )

        if not response.response_stages:
            raise ValueError(
                f"No response stages attached to {channel.code}"
            )

        old_sensitivity = float(
            response.instrument_sensitivity.value
        )

        calib_pa_per_count = float(calib_pa_per_count)

        if calib_pa_per_count <= 0:
            raise ValueError(
                f"Calibration must be positive for {waveform_channel}; "
                f"got {calib_pa_per_count}"
            )

        new_sensitivity = 1.0 / calib_pa_per_count
        scale_factor = new_sensitivity / old_sensitivity

        first_stage = response.response_stages[0]
        old_stage_gain = float(first_stage.stage_gain)
        first_stage.stage_gain = old_stage_gain * scale_factor

        response.instrument_sensitivity.value = new_sensitivity
        response.instrument_sensitivity.frequency = 1.0
        response.instrument_sensitivity.input_units = "Pa"
        response.instrument_sensitivity.output_units = "count"

        print(
            f"{waveform_channel} -> {xml_channel}: "
            f"{old_sensitivity:.6g} to "
            f"{new_sensitivity:.6g} count/Pa; "
            f"stage-1 gain scaled by {scale_factor:.6g}"
        )

    return calibrated


inventory_event_antelope = make_antelope_calibrated_inventory(
    inventory_event,
    network=BCHH_NETWORK,
    station=BCHH_STATION,
    location=BCHH_LOCATION,
    channel_calib_pa_per_count=pressure_calib_pa_per_count,
    waveform_to_xml_channel=WAVEFORM_TO_XML_CHANNEL,
    time=EXPLOSION_TIME,
)


print("\nEmpirically calibrated pressure responses:")

for channel in ["DD1", "DD2", "DD3"]:
    seed_id = (
        f"{BCHH_NETWORK}."
        f"{BCHH_STATION}."
        f"{BCHH_LOCATION}."
        f"{channel}"
    )

    response = inventory_event_antelope.get_response(
        seed_id,
        EXPLOSION_TIME,
    )

    sensitivity = response.instrument_sensitivity

    print(
        channel,
        sensitivity.value,
        sensitivity.output_units,
        "/",
        sensitivity.input_units,
    )


# Part II — corrected standalone infraBSU/NRL functions

The following functions replace the relevant experimental functionality from `build.py` and `infrabsu.py` inside this notebook.

Important corrections:

- infraBSU input units are explicitly `Pa`;
- the infraBSU sensor sensitivity is `4.6e-5 V/Pa`;
- Centaur 40 Vpp is used for BCHH, yielding `18.4 count/Pa`;
- Centaur 1 Vpp would yield `736 count/Pa`;
- there is no erroneous `18400 count/Pa` fallback;
- each requested channel receives an independent deep copy of the response;
- coordinates and epoch times are explicit arguments.


## 7. General inventory builder (corrected replacement for `build.NRL2inventory`)


In [ ]:
def _make_nrl(nrl_path: Optional[Path | str] = None) -> NRL:
    """Open a local NRL when supplied; otherwise use the remote NRL service."""
    if nrl_path is not None and Path(nrl_path).is_dir():
        return NRL(str(nrl_path))
    return NRL("https://ds.iris.edu/NRL/")


def _centaur_keys(vpp: float, sample_rate_hz: float) -> list[str]:
    if float(vpp) == 40.0:
        gain_key = "40 Vpp (1)"
    elif float(vpp) == 1.0:
        gain_key = "1 Vpp (40)"
    else:
        raise ValueError(f"Centaur Vpp must be 1 or 40, got {vpp}")
    return [
        "Nanometrics", "Centaur", gain_key,
        "Off", "Linear phase", f"{int(sample_rate_hz)}",
    ]


def _sensor_keys(sensor: str) -> list[str]:
    name = sensor.strip().lower()
    if name == "infrabsu":
        raise KeyError(
            "infraBSU is not requested directly from ObsPy's NRL sensor tree. "
            "Use the sensor-template + Centaur-stage builder instead."
        )
    if name in {"tcp", "trillium compact", "trillium compact 120"}:
        return ["Nanometrics", "Trillium Compact 120 (Vault, Posthole, OBS)", "754 V/m/s"]
    if name.startswith("chap"):
        return ["Chaparral Physics", "25", "Low: 0.4 V/Pa"]
    raise ValueError(f"Unsupported sensor: {sensor}")


def responses_to_inventory(
    *,
    network: str,
    station: str,
    location: str,
    channel_codes: Iterable[str],
    response: Response,
    sample_rate_hz: float,
    latitude: float,
    longitude: float,
    elevation_m: float,
    depth_m: float,
    start_date: UTCDateTime,
    end_date: UTCDateTime,
    site_name: str = "",
    source: str = "standalone NRL builder",
) -> Inventory:
    """Create a one-station Inventory and deep-copy the response per channel."""
    channels = []
    for channel_code in channel_codes:
        channels.append(Channel(
            code=str(channel_code),
            location_code=str(location),
            latitude=float(latitude),
            longitude=float(longitude),
            elevation=float(elevation_m),
            depth=float(depth_m),
            sample_rate=float(sample_rate_hz),
            start_date=start_date,
            end_date=end_date,
            response=copy.deepcopy(response),
        ))

    sta = Station(
        code=str(station),
        latitude=float(latitude),
        longitude=float(longitude),
        elevation=float(elevation_m),
        creation_date=start_date,
        site=Site(name=site_name or station),
        channels=channels,
    )
    net = Network(code=str(network), stations=[sta], start_date=start_date, end_date=end_date)
    return Inventory(networks=[net], source=source)



def _normalize_unit_name(unit) -> str | None:
    if unit is None:
        return None
    name = getattr(unit, "name", unit if isinstance(unit, str) else None)
    if name is None:
        return None
    text = str(name).strip()
    if text.upper() in {"COUNT", "COUNTS"}:
        return "count"
    if text.upper() == "PA":
        return "Pa"
    if text.upper() == "V":
        return "V"
    return text


def _stage_gain_value(stage) -> float:
    gain = getattr(stage, "stage_gain", None)
    if gain is None:
        return 1.0
    return float(getattr(gain, "value", gain))


def _recalculate_overall_sensitivity(
    response: Response,
    frequency_hz: float = 1.0,
) -> None:
    stages = list(response.response_stages or [])
    if not stages:
        raise ValueError("Response has no stages")
    response.instrument_sensitivity = InstrumentSensitivity(
        value=float(np.prod([_stage_gain_value(stage) for stage in stages])),
        frequency=float(frequency_hz),
        input_units=_normalize_unit_name(stages[0].input_units) or "Pa",
        output_units=_normalize_unit_name(stages[-1].output_units) or "count",
    )


def _load_infrabsu_sensor_response(template_path: Path | str) -> Response:
    path = Path(template_path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    inventory = read_inventory(str(path))
    channels = [
        channel
        for network in inventory
        for station in network
        for channel in station
    ]
    if len(channels) != 1:
        raise ValueError(
            f"Expected one channel in infraBSU template {path}; "
            f"found {len(channels)}"
        )
    if channels[0].response is None:
        raise ValueError(f"No response found in infraBSU template {path}")
    return copy.deepcopy(channels[0].response)


def _build_infrabsu_centaur_response(
    *,
    sample_rate_hz: float,
    vpp: float,
    template_path: Path | str,
    nrl_path: Optional[Path | str] = None,
) -> Response:
    """Combine the infraBSU sensor template with Centaur response stages."""
    sensor_response = _load_infrabsu_sensor_response(template_path)
    sensor_stages = copy.deepcopy(sensor_response.response_stages or [])
    if not sensor_stages:
        raise ValueError("infraBSU sensor template contains no response stages")

    nrl = _make_nrl(nrl_path)
    dummy_complete = nrl.get_response(
        sensor_keys=[
            "Nanometrics",
            "Trillium Compact 120 (Vault, Posthole, OBS)",
            "754 V/m/s",
        ],
        datalogger_keys=_centaur_keys(vpp, sample_rate_hz),
    )
    dummy_stages = copy.deepcopy(dummy_complete.response_stages or [])
    if len(dummy_stages) < 2:
        raise ValueError(
            "Dummy sensor + Centaur response does not contain separable stages"
        )

    # First stage is the dummy seismometer; all later stages belong to Centaur.
    centaur_stages = dummy_stages[1:]
    combined = Response(response_stages=sensor_stages + centaur_stages)
    _recalculate_overall_sensitivity(combined, frequency_hz=1.0)
    return combined


def NRL2inventory(
    nrl_path,
    net,
    sta,
    loc,
    chans,
    datalogger="Centaur",
    sensor="TCP",
    Vpp=40,
    fsamp=100.0,
    lat=0.0,
    lon=0.0,
    elev=0.0,
    depth=0.0,
    sitename="",
    ondate=UTCDateTime(1970, 1, 1),
    offdate=UTCDateTime(2100, 1, 1),
    infrabsu_template_path: Optional[Path | str] = None,
) -> Inventory:
    """Standalone NRL builder with a special infraBSU path."""
    if datalogger.lower() != "centaur":
        raise ValueError("Only the Nanometrics Centaur is supported here")

    channel_codes = [chans] if isinstance(chans, str) else list(chans)
    sensor_name = sensor.strip().lower()

    if sensor_name == "infrabsu":
        if infrabsu_template_path is None:
            raise ValueError(
                "infrabsu_template_path is required because this ObsPy NRL "
                "tree does not expose JeffreyBJohnson/infraBSU"
            )
        response = _build_infrabsu_centaur_response(
            sample_rate_hz=fsamp,
            vpp=Vpp,
            template_path=infrabsu_template_path,
            nrl_path=nrl_path,
        )
        response_source = (
            f"infraBSU sensor template + NRL Centaur {Vpp:g} Vpp"
        )
    else:
        nrl = _make_nrl(nrl_path)
        response = nrl.get_response(
            sensor_keys=_sensor_keys(sensor),
            datalogger_keys=_centaur_keys(Vpp, fsamp),
        )
        response_source = f"NRL: {sensor} + {datalogger} {Vpp:g} Vpp"

    return responses_to_inventory(
        network=net,
        station=sta,
        location=loc,
        channel_codes=channel_codes,
        response=response,
        sample_rate_hz=fsamp,
        latitude=lat,
        longitude=lon,
        elevation_m=elev,
        depth_m=depth,
        start_date=ondate,
        end_date=offdate,
        site_name=sitename,
        source=response_source,
    )


## 8. infraBSU-specific builder (corrected replacement for `infrabsu.py`)

The direct NRL combination is preferred. A sensor-template function is retained for explicit inspection or offline reuse, but it does not depend on FLOVOpy.


In [ ]:
DEFAULT_INFRABSU_SENSOR_URL = (
    "https://service.iris.edu/irisws/nrl/1/combine"
    "?instconfig=sensor_JeffreyBJohnson_infraBSU_LP21_SG0.000046_STairPressure"
    "&format=stationxml"
)


def get_infrabsu_sensor_template(
    local_filename: Path | str = "infraBSU_sensor.xml",
    url: str = DEFAULT_INFRABSU_SENSOR_URL,
    timeout_s: float = 60.0,
    overwrite: bool = False,
) -> Path:
    """Download/cache the sensor-only infraBSU StationXML without FLOVOpy dependencies."""
    path = Path(local_filename).expanduser().resolve()
    if path.exists() and not overwrite:
        return path

    import requests
    response = requests.get(url, timeout=timeout_s)
    response.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(response.content)

    # Verify that the downloaded file is readable StationXML.
    read_inventory(str(path))
    return path


def get_infrabsu_centaur(
    *,
    fsamp: float,
    vpp: float,
    network: str,
    station: str,
    location: str,
    channels: Iterable[str] | str,
    latitude: float,
    longitude: float,
    elevation: float,
    depth: float,
    start_date: UTCDateTime,
    end_date: UTCDateTime,
    template_path: Path | str,
    nrl_path: Optional[Path | str] = None,
) -> Inventory:
    """Build an infraBSU + Centaur inventory directly from the NRL.

    For the BCHH 2016 deployment use `vpp=40`, which should produce an overall
    sensitivity of 18.4 count/Pa at 1 Hz.
    """
    channel_codes = [channels] if isinstance(channels, str) else list(channels)
    return NRL2inventory(
        nrl_path=nrl_path,
        net=network,
        sta=station,
        loc=location,
        chans=channel_codes,
        datalogger="Centaur",
        sensor="infrabsu",
        Vpp=vpp,
        fsamp=fsamp,
        lat=latitude,
        lon=longitude,
        elev=elevation,
        depth=depth,
        sitename=station,
        ondate=start_date,
        offdate=end_date,
        infrabsu_template_path=template_path,
    )


## 9. Build 40 Vpp and 1 Vpp NRL responses using inventory coordinates

This cell gets the coordinates from `inventory_event`, not from hardcoded constants.


In [ ]:
# Show what this ObsPy NRL client actually exposes.
nrl_diagnostic = _make_nrl(NRL_PATH)
visible_sensor_manufacturers = list(nrl_diagnostic.sensors.keys())
print("Visible NRL sensor manufacturers:", len(visible_sensor_manufacturers))
print("JeffreyBJohnson visible?", "JeffreyBJohnson" in visible_sensor_manufacturers)
print("Possible infrasound-related names:")
for manufacturer in visible_sensor_manufacturers:
    text = str(manufacturer).lower()
    if any(term in text for term in ("johnson", "infra", "bsu")):
        print("  ", manufacturer)


In [ ]:
def station_coordinates_from_inventory(
    inventory: Inventory,
    *,
    network: str,
    station: str,
    location: str,
    channel: str,
    time: UTCDateTime,
) -> dict:
    selected = inventory.select(
        network=network,
        station=station,
        location=location,
        channel=channel,
        time=time,
    )
    channels = [cha for net in selected for sta in net for cha in sta]
    if len(channels) != 1:
        raise ValueError(f"Expected one channel for coordinate lookup, found {len(channels)}")
    cha = channels[0]
    return {
        "latitude": float(cha.latitude),
        "longitude": float(cha.longitude),
        "elevation": float(cha.elevation),
        "depth": float(cha.depth),
        "sample_rate": float(cha.sample_rate),
    }


bchh_coordinates = station_coordinates_from_inventory(
    inventory_event,
    network=BCHH_NETWORK,
    station=BCHH_STATION,
    location=BCHH_LOCATION,
    channel="DD1",
    time=EXPLOSION_TIME,
)
print(bchh_coordinates)

inv_infrabsu_nrl_40vpp = get_infrabsu_centaur(
    fsamp=bchh_coordinates["sample_rate"],
    vpp=40,
    network=BCHH_NETWORK,
    station=BCHH_STATION,
    location=BCHH_LOCATION,
    channels=["DD1", "DD2", "DD3"],
    latitude=bchh_coordinates["latitude"],
    longitude=bchh_coordinates["longitude"],
    elevation=bchh_coordinates["elevation"],
    depth=bchh_coordinates["depth"],
    start_date=READ_START,
    end_date=READ_END,
    template_path=INFRABSU_TEMPLATE_FILE,
    nrl_path=NRL_PATH,
)

inv_infrabsu_nrl_1vpp = get_infrabsu_centaur(
    fsamp=bchh_coordinates["sample_rate"],
    vpp=1,
    network=BCHH_NETWORK,
    station=BCHH_STATION,
    location=BCHH_LOCATION,
    channels=["DD1", "DD2", "DD3"],
    latitude=bchh_coordinates["latitude"],
    longitude=bchh_coordinates["longitude"],
    elevation=bchh_coordinates["elevation"],
    depth=bchh_coordinates["depth"],
    start_date=READ_START,
    end_date=READ_END,
    template_path=INFRABSU_TEMPLATE_FILE,
    nrl_path=NRL_PATH,
)


### Empirical calibration versus nominal 40 Vpp response

This table shows why the nominal response cannot be used as the amplitude calibration for this deployment.


In [ ]:
nominal_count_per_pa = 18.4
calibration_comparison_df = active_pressure_calibration_df[
    ["waveform_channel", "calib", "count_per_unit"]
].copy()
calibration_comparison_df = calibration_comparison_df.rename(
    columns={
        "calib": "empirical_pa_per_count",
        "count_per_unit": "empirical_count_per_pa",
    }
)
calibration_comparison_df["nominal_count_per_pa"] = nominal_count_per_pa
calibration_comparison_df["empirical_to_nominal_pa_per_count"] = (
    calibration_comparison_df["empirical_pa_per_count"]
    / (1.0 / nominal_count_per_pa)
)
display(calibration_comparison_df)


## 10. Stage-by-stage response inspection


In [ ]:
def response_stage_table(inventory: Inventory, seed_id: str, time: UTCDateTime) -> pd.DataFrame:
    response = inventory.get_response(seed_id, time)
    sensitivity = response.instrument_sensitivity
    rows = [{
        "stage": 0,
        "type": "Overall sensitivity",
        "gain": float(sensitivity.value),
        "gain_frequency_hz": float(sensitivity.frequency),
        "input_units": str(sensitivity.input_units),
        "output_units": str(sensitivity.output_units),
    }]
    for stage in response.response_stages:
        rows.append({
            "stage": int(stage.stage_sequence_number),
            "type": type(stage).__name__,
            "gain": None if stage.stage_gain is None else float(stage.stage_gain),
            "gain_frequency_hz": None if stage.stage_gain_frequency is None else float(stage.stage_gain_frequency),
            "input_units": str(stage.input_units),
            "output_units": str(stage.output_units),
        })
    return pd.DataFrame(rows)


for label, inventory in [
    ("Event StationXML", inventory_event),
    ("NRL 40 Vpp", inv_infrabsu_nrl_40vpp),
    ("NRL 1 Vpp", inv_infrabsu_nrl_1vpp),
]:
    print(f"\n{label}")
    display(response_stage_table(inventory, "1R.BCHH.10.DD1", EXPLOSION_TIME))


## 11. Frequency-response comparison

This compares amplitude and phase rather than only the nominal sensitivity at 1 Hz.


In [ ]:
def evaluate_response_at_frequencies(
    inventory: Inventory,
    seed_id: str,
    time: UTCDateTime,
    target_frequencies_hz: Iterable[float],
    sample_rate_hz: float,
    nfft: int = 2 ** 18,
) -> pd.DataFrame:
    response = inventory.get_response(seed_id, time)
    complex_response, frequencies = response.get_evalresp_response(
        t_samp=1.0 / sample_rate_hz,
        nfft=nfft,
        output="DEF",
    )
    rows = []
    for target in target_frequencies_hz:
        index = int(np.argmin(np.abs(frequencies - float(target))))
        value = complex_response[index]
        rows.append({
            "frequency_hz": float(frequencies[index]),
            "amplitude_count_per_pa": float(np.abs(value)),
            "phase_deg": float(np.degrees(np.angle(value))),
        })
    return pd.DataFrame(rows)


TEST_FREQUENCIES_HZ = [0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2, 5, 10, 20, 50, 80, 100]
comparison_tables = []
for label, inventory in [
    ("event_stationxml", inventory_event),
    ("nrl_40vpp", inv_infrabsu_nrl_40vpp),
    ("nrl_1vpp", inv_infrabsu_nrl_1vpp),
]:
    table = evaluate_response_at_frequencies(
        inventory,
        "1R.BCHH.10.DD1",
        EXPLOSION_TIME,
        TEST_FREQUENCIES_HZ,
        bchh_coordinates["sample_rate"],
    )
    table.insert(0, "source", label)
    comparison_tables.append(table)

nrl_response_comparison_df = pd.concat(comparison_tables, ignore_index=True)
nrl_response_comparison_df.to_csv(NRL_COMPARISON_CSV, index=False)
display(nrl_response_comparison_df)


In [ ]:
amplitude_pivot = nrl_response_comparison_df.pivot(
    index="frequency_hz",
    columns="source",
    values="amplitude_count_per_pa",
)
amplitude_pivot["nrl_40vpp_to_event"] = (
    amplitude_pivot["nrl_40vpp"] / amplitude_pivot["event_stationxml"]
)
amplitude_pivot["nrl_1vpp_to_event"] = (
    amplitude_pivot["nrl_1vpp"] / amplitude_pivot["event_stationxml"]
)
display(amplitude_pivot)


## 12. Data-domain comparison: sensitivity-only pressure

This test should give identical waveforms for the event StationXML and NRL 40 Vpp inventories when their overall sensitivities agree. The 1 Vpp result should be smaller by a factor of 40.


In [ ]:
def pressure_from_inventory_sensitivity(
    raw_trace,
    inventory: Inventory,
    inventory_seed_id: str,
    baseline_start: UTCDateTime,
    baseline_end: UTCDateTime,
):
    response = inventory.get_response(inventory_seed_id, raw_trace.stats.starttime)
    sensitivity = response.instrument_sensitivity
    if str(sensitivity.input_units).lower() != "pa":
        raise ValueError(f"Expected Pa, got {sensitivity.input_units}")

    out = raw_trace.copy()
    out.data = np.asarray(out.data, dtype=np.float64) / float(sensitivity.value)
    out.data -= median_in_window(out, baseline_start, baseline_end)
    out.stats.units = "Pa"
    return out


data_domain_rows = []
for waveform_channel, xml_channel in [("HD1", "DD1"), ("HD2", "DD2"), ("HD3", "DD3")]:
    raw_trace = st_raw.select(channel=waveform_channel)[0]
    for label, inventory in [
        ("event_stationxml", inventory_event),
        ("nrl_40vpp", inv_infrabsu_nrl_40vpp),
        ("nrl_1vpp", inv_infrabsu_nrl_1vpp),
    ]:
        corrected = pressure_from_inventory_sensitivity(
            raw_trace,
            inventory,
            f"{BCHH_NETWORK}.{BCHH_STATION}.{BCHH_LOCATION}.{xml_channel}",
            PRESSURE_BASELINE_START,
            PRESSURE_BASELINE_END,
        ).trim(CHECK_START, CHECK_END)
        data = np.asarray(corrected.data, dtype=np.float64)
        data_domain_rows.append({
            "channel": waveform_channel,
            "response_source": label,
            "minimum_pa": float(np.nanmin(data)),
            "maximum_pa": float(np.nanmax(data)),
            "peak_absolute_pa": float(np.nanmax(np.abs(data))),
            "peak_to_peak_pa": float(np.nanmax(data) - np.nanmin(data)),
        })

data_domain_comparison_df = pd.DataFrame(data_domain_rows)
display(data_domain_comparison_df)


## 13. Optional full-deconvolution comparison for infraBSU

This is deliberately diagnostic, not the production method. It demonstrates how strongly the result depends on low-frequency stabilization. Do not run with both `pre_filt=None` and `water_level=None`.


In [ ]:
def full_infrasound_deconvolution(
    raw_trace,
    inventory: Inventory,
    inventory_seed_id: str,
    *,
    pre_filt=(0.05, 0.1, 80.0, 100.0),
    water_level_db=60.0,
):
    out = raw_trace.copy()
    parts = inventory_seed_id.split(".")
    out.stats.network, out.stats.station, out.stats.location, out.stats.channel = parts
    out.detrend("demean")
    out.detrend("linear")
    out.taper(max_percentage=0.05, type="cosine")
    out.remove_response(
        inventory=inventory,
        output="DEF",
        pre_filt=pre_filt,
        water_level=water_level_db,
        zero_mean=False,
        taper=False,
    )
    out.stats.units = "Pa"
    return out


# Example for HD1. Change RUN_FULL_INFRASOUND_DECONVOLUTION to True to execute.
RUN_FULL_INFRASOUND_DECONVOLUTION = False
if RUN_FULL_INFRASOUND_DECONVOLUTION:
    raw_hd1 = st_raw.select(channel="HD1")[0]
    full_event = full_infrasound_deconvolution(
        raw_hd1, inventory_event, "1R.BCHH.10.DD1"
    ).trim(CHECK_START, CHECK_END)
    full_nrl = full_infrasound_deconvolution(
        raw_hd1, inv_infrabsu_nrl_40vpp, "1R.BCHH.10.DD1"
    ).trim(CHECK_START, CHECK_END)

    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(full_event.times(), full_event.data, label="Event StationXML", linewidth=0.8)
    ax.plot(full_nrl.times(), full_nrl.data, label="NRL 40 Vpp", linewidth=0.8, alpha=0.75)
    ax.set_xlabel("Time from comparison-window start (s)")
    ax.set_ylabel("Pressure (Pa)")
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    plt.show()


## 14. Expected interpretation

A successful run should show:

- event StationXML infraBSU sensitivity: approximately **18.4 count/Pa**;
- NRL 40 Vpp sensitivity: approximately **18.4 count/Pa**;
- NRL 1 Vpp sensitivity: approximately **736 count/Pa**;
- NRL 40 Vpp / event response amplitude ratio near 1 through the useful passband;
- sensitivity-corrected event and NRL 40 Vpp waveforms numerically identical apart from floating-point roundoff;
- the 1 Vpp pressure waveform smaller by a factor of approximately 40.

The output bundle in `outputs/response_correction/` can then be consumed by the figure-generation notebook without repeating calibration experiments.


## Validation status

This notebook is intentionally outside the canonical production sequence. Any
scientific decision adopted from these experiments should be recorded in the
manuscript methods, the repository README, or Notebook 01 provenance metadata.
